# ETL - Buscador de Vagas Ocultas de Startups

Pipeline que descobre vagas escondidas em plataformas de ATS como InHire.

Funcionalidades:

- Descobre empresas automaticamente
- Extrai vagas carregadas via JavaScript
- Filtra vagas da área de dados
- Filtra vagas remotas
- Compara com execuções anteriores
- Marca vagas novas
- Salva dataset final


## Instalar dependências

Execute apenas uma vez.

In [ ]:
!pip install playwright pandas openpyxl requests
!playwright install

## Importar bibliotecas

In [ ]:
import pandas as pd
import requests
from playwright.sync_api import sync_playwright
from datetime import datetime
import os
import re

## Configurações do ETL

In [ ]:
DATA_ETL = datetime.now().strftime('%Y-%m-%d')

OUTPUT_FILE = "vagas_startups_dados.xlsx"

KEYWORDS_DADOS = [
    "data",
    "analytics",
    "machine learning",
    "ai",
    "cientista",
    "analista de dados",
    "data engineer",
    "business intelligence"
]

## Descobrir empresas usando InHire

In [ ]:
def descobrir_empresas():

    empresas = []

    queries = [
        "site:inhire.app vagas",
        "site:inhire.app jobs",
        "site:inhire.app careers"
    ]

    for q in queries:

        url = f"https://www.google.com/search?q={q}&num=100"

        html = requests.get(url, headers={"User-Agent":"Mozilla/5.0"}).text

        encontrados = re.findall(r"https://([a-z0-9-]+)\.inhire\.app", html)

        empresas.extend(encontrados)

    empresas = list(set(empresas))

    return empresas

## Coletar vagas via browser real

In [ ]:
def coletar_vagas(empresas):

    vagas = []

    with sync_playwright() as p:

        browser = p.chromium.launch(headless=True)

        for empresa in empresas:

            try:

                url = f"https://{empresa}.inhire.app/vagas"

                page = browser.new_page()

                page.goto(url, timeout=60000)

                page.wait_for_timeout(4000)

                cards = page.query_selector_all("a[href*='jobs'], a[href*='vagas']")

                for card in cards:

                    titulo = card.inner_text()

                    link = card.get_attribute("href")

                    if not link.startswith("http"):

                        link = f"https://{empresa}.inhire.app" + link

                    vagas.append({
                        "empresa": empresa,
                        "vaga": titulo,
                        "local": "Remoto",
                        "link": link
                    })

                page.close()

            except:
                pass

        browser.close()

    return vagas

## Filtrar vagas da área de dados

In [ ]:
def filtrar_dados(df):

    pattern = '|'.join(KEYWORDS_DADOS)

    df = df[df['vaga'].str.lower().str.contains(pattern, na=False)]

    return df

## Filtrar vagas remotas

In [ ]:
def filtrar_remoto(df):

    remoto_keywords = ['remote','remoto','anywhere']

    df = df[df['vaga'].str.lower().str.contains('|'.join(remoto_keywords), na=False)]

    return df

## Detectar vagas novas

In [ ]:
def detectar_novas(df):

    if not os.path.exists(OUTPUT_FILE):

        df['nova_vaga'] = True

        return df

    antigo = pd.read_excel(OUTPUT_FILE)

    antigo_links = set(antigo['link'])

    df['nova_vaga'] = ~df['link'].isin(antigo_links)

    return df

## Pipeline principal

In [ ]:
print("Descobrindo empresas...")

empresas = descobrir_empresas()

print("Empresas encontradas:", len(empresas))

print("Coletando vagas...")

vagas = coletar_vagas(empresas)

df = pd.DataFrame(vagas)

print("Total vagas coletadas:", len(df))

df = filtrar_dados(df)

df = filtrar_remoto(df)

df['data_etl'] = DATA_ETL

df = detectar_novas(df)

df.to_excel(OUTPUT_FILE, index=False)

print("Arquivo salvo:", OUTPUT_FILE)

df.head()